# Legacy binary WESAD notebook

This notebook remains as a historical binary experiment and must not produce, validate, or export the four-level physiological stress model. WESAD wrist signals are useful only as a modality reference (EDA 4 Hz, BVP 64 Hz, ACC 32 Hz). Its baseline/stress/amusement/meditation protocol labels are not a scientifically defensible mapping to RELAXED, LOW_STRESS, MODERATE_STRESS, and HIGH_STRESS. Use `train_model.py` with an approved four-class labelled physiological manifest for future training.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')
# Install required packages
!pip install -q neurokit2 xgboost imbalanced-learn

# ==============================
# Imports
# ==============================

import os
import glob
import pickle
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import skew, kurtosis

import neurokit2 as nk

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("All packages imported successfully!")
# ==========================================
# WESAD Dataset Path and Subject List
# ==========================================

BASE_PATH = "/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD"

# WESAD subjects (S12 does not exist)
subjects = sorted([
    s for s in os.listdir(BASE_PATH)
    if s.startswith("S")
])

print("Subjects Found:")
print(subjects)

print("\nTotal Subjects:", len(subjects))

# Sampling rates (Wrist signals)
EDA_FS = 4      # EDA/GSR
BVP_FS = 64     # BVP/PPG
ACC_FS = 32     # IMU
LABEL_FS = 700  # WESAD labels

# Window settings
WINDOW_SECONDS = 30
STEP_SECONDS = 15   # 50% overlap

print("\nSampling Rates:")
print("EDA:", EDA_FS, "Hz")
print("BVP:", BVP_FS, "Hz")
print("ACC:", ACC_FS, "Hz")
print("Labels:", LABEL_FS, "Hz")

print("\nWindow Configuration:")
print("Window Length:", WINDOW_SECONDS, "seconds")
print("Overlap:", 50, "%")
print("Step Size:", STEP_SECONDS, "seconds")
# ==========================================
# Load One Subject from WESAD
# ==========================================

def load_subject(subject_id):
    """
    Load one WESAD subject and extract wrist signals.
    
    Returns:
        eda    : Wrist EDA (4 Hz)
        bvp    : Wrist BVP (64 Hz)
        acc    : Wrist ACC (32 Hz, 3 axes)
        labels : WESAD labels (700 Hz)
    """

    subject_path = os.path.join(
        BASE_PATH,
        subject_id,
        f"{subject_id}.pkl"
    )

    with open(subject_path, "rb") as f:
        data = pickle.load(
            f,
            encoding="latin1"
        )

    wrist = data["signal"]["wrist"]

    eda = wrist["EDA"].flatten()

    bvp = wrist["BVP"].flatten()

    acc = wrist["ACC"]

    labels = data["label"]

    return eda, bvp, acc, labels


# ==========================================
# Test on Subject S2
# ==========================================

eda, bvp, acc, labels = load_subject("S2")

print("EDA Shape:", eda.shape)
print("BVP Shape:", bvp.shape)
print("ACC Shape:", acc.shape)
print("Labels Shape:", labels.shape)

print("\nEDA Sampling Rate:", EDA_FS, "Hz")
print("BVP Sampling Rate:", BVP_FS, "Hz")
print("ACC Sampling Rate:", ACC_FS, "Hz")
print("Label Sampling Rate:", LABEL_FS, "Hz")
# ==========================================
# Binary Label Mapping
# ==========================================

def map_label(label):
    """
    WESAD Binary Mapping

    Non-Stress (0):
        Baseline (1)
        Amusement (3)
        Meditation (4)

    Stress (1):
        Stress (2)

    Ignore:
        0,5,6,7
    """

    if label in [1, 3, 4]:
        return 0

    elif label == 2:
        return 1

    else:
        return -1


# Test
test_labels = [0,1,2,3,4,5,6,7]

mapped = [map_label(x) for x in test_labels]

print(pd.DataFrame({
    "Original": test_labels,
    "Mapped": mapped
}))
# ==========================================
# EDA Features (4 Hz) - Final Version
# ==========================================

def extract_eda_features(eda_window):

    try:

        cleaned = nk.eda_clean(
            eda_window,
            sampling_rate=EDA_FS
        )

        decomposed = nk.eda_phasic(
            cleaned,
            sampling_rate=EDA_FS
        )

        tonic = decomposed["EDA_Tonic"]

        phasic = decomposed["EDA_Phasic"]

        _, peak_info = nk.eda_peaks(
            phasic,
            sampling_rate=EDA_FS
        )

        scr_count = len(
            peak_info["SCR_Peaks"]
        )

        # SCR Amplitude
        if scr_count > 0:
            scr_amp_mean = np.nanmean(
                peak_info["SCR_Amplitude"]
            )
        else:
            scr_amp_mean = 0


        # SCR Rise Time
        if (
            "SCR_RiseTime" in peak_info
            and len(peak_info["SCR_RiseTime"]) > 0
        ):
            scr_rise_mean = np.nanmean(
                peak_info["SCR_RiseTime"]
            )
        else:
            scr_rise_mean = 0


        # SCR Recovery Time
        if (
            "SCR_RecoveryTime" in peak_info
            and len(peak_info["SCR_RecoveryTime"]) > 0
        ):
            scr_recovery_mean = np.nanmean(
                peak_info["SCR_RecoveryTime"]
            )
        else:
            scr_recovery_mean = 0


        features = {

            "eda_mean":
                np.mean(cleaned),

            "eda_std":
                np.std(cleaned),

            "eda_slope":
                np.polyfit(
                    np.arange(len(cleaned)),
                    cleaned,
                    1
                )[0],

            "scl_mean":
                np.mean(tonic),

            "phasic_mean":
                np.mean(phasic),

            "scr_count":
                scr_count,

            "scr_amp_mean":
                scr_amp_mean,

            "scr_rise_mean":
                scr_rise_mean,

            "scr_recovery_mean":
                scr_recovery_mean
        }

        return features

    except:

        return {

            "eda_mean": np.nan,

            "eda_std": np.nan,

            "eda_slope": np.nan,

            "scl_mean": np.nan,

            "phasic_mean": np.nan,

            "scr_count": np.nan,

            "scr_amp_mean": np.nan,

            "scr_rise_mean": np.nan,

            "scr_recovery_mean": np.nan
        }
# ==========================================
# PPG Features using NeuroKit2
# ==========================================

invalid_hrv_count = 0
total_hrv_windows = 0


def extract_ppg_features(ppg_window):

    global invalid_hrv_count
    global total_hrv_windows

    total_hrv_windows += 1

    try:

        _, info = nk.ppg_process(
            ppg_window,
            sampling_rate=BVP_FS
        )

        peaks = info["PPG_Peaks"]

        if len(peaks) < 3:
            raise Exception()

        hrv = nk.hrv(
            peaks,
            sampling_rate=BVP_FS,
            show=False
        )

        ibi = np.diff(peaks)

        hr = 60 / (
            np.mean(ibi) / BVP_FS
        )

        return {

            "hr": hr,

            "rmssd":
                hrv["HRV_RMSSD"].iloc[0],

            "sdnn":
                hrv["HRV_SDNN"].iloc[0],

            "pnn50":
                hrv["HRV_pNN50"].iloc[0],

            "ibi_mean":
                np.mean(ibi),

            "ibi_std":
                np.std(ibi)

        }

    except:

        invalid_hrv_count += 1

        return {

            "hr": np.nan,

            "rmssd": np.nan,

            "sdnn": np.nan,

            "pnn50": np.nan,

            "ibi_mean": np.nan,

            "ibi_std": np.nan

        }


# Test
print(extract_ppg_features(
    bvp[:1920]
))
# ==========================================
# IMU Features
# ==========================================

def extract_imu_features(acc_window):

    magnitude = np.sqrt(
        np.sum(acc_window**2, axis=1)
    )

    jerk = np.diff(magnitude)

    return {

        "imu_mag_mean":
            np.mean(magnitude),

        "imu_mag_std":
            np.std(magnitude),

        "imu_energy":
            np.sum(magnitude**2),

        "imu_jerk_mean":
            np.mean(jerk),

        "imu_jerk_std":
            np.std(jerk),

        "imu_var_x":
            np.var(acc_window[:,0]),

        "imu_var_y":
            np.var(acc_window[:,1]),

        "imu_var_z":
            np.var(acc_window[:,2])

    }


# Test
print(extract_imu_features(
    acc[:960]
))
# ==========================================
# Window Generator
# ==========================================

def generate_windows(signal,
                     fs,
                     window_sec=30,
                     step_sec=15):

    window = fs * window_sec

    step = fs * step_sec

    for start in range(
        0,
        len(signal) - window + 1,
        step
    ):

        end = start + window

        yield signal[start:end]
# ==========================================
# Build Subject Dataset
# ==========================================

def build_subject_dataset(subject_id):

    X = []
    y = []
    groups = []

    eda, bvp, acc, labels = load_subject(
        subject_id
    )

    eda_windows = list(
        generate_windows(
            eda,
            EDA_FS
        )
    )

    bvp_windows = list(
        generate_windows(
            bvp,
            BVP_FS
        )
    )

    acc_windows = list(
        generate_windows(
            acc,
            ACC_FS
        )
    )

    label_windows = list(
        generate_windows(
            labels,
            LABEL_FS
        )
    )

    n = min(

        len(eda_windows),

        len(bvp_windows),

        len(acc_windows),

        len(label_windows)

    )

    for i in range(n):

        mapped = np.array([

            map_label(l)

            for l in label_windows[i]

        ])

        mapped = mapped[
            mapped != -1
        ]

        if len(mapped) == 0:
            continue

        label = np.bincount(
            mapped
        ).argmax()

        features = {}

        features.update(
            extract_eda_features(
                eda_windows[i]
            )
        )

        features.update(
            extract_ppg_features(
                bvp_windows[i]
            )
        )

        features.update(
            extract_imu_features(
                acc_windows[i]
            )
        )

        X.append(features)

        y.append(label)

        groups.append(subject_id)

    return X, y, groups
# ==========================================
# Build Full Dataset
# ==========================================

all_X = []
all_y = []
all_groups = []

for subject in subjects:

    print(
        "Processing",
        subject
    )

    X_sub, y_sub, g_sub = (
        build_subject_dataset(
            subject
        )
    )

    all_X.extend(X_sub)

    all_y.extend(y_sub)

    all_groups.extend(g_sub)


df = pd.DataFrame(all_X)

df["label"] = all_y

df["subject_id"] = all_groups


print("\nBefore NaN Removal:")
print(df.shape)

df = df.dropna()

print("\nAfter NaN Removal:")
print(df.shape)


invalid_hrv_percentage = (
    invalid_hrv_count
    / total_hrv_windows
) * 100

print(
    "\nInvalid HRV Windows (%):",
    round(
        invalid_hrv_percentage,
        2
    )
)

print("\nClass Distribution:")

print(
    df["label"].value_counts()
)

print("\nSubjects:")

print(
    df["subject_id"].nunique()
)

df.head()
X = df.drop(
    ["label","subject_id"],
    axis=1
)

y = df["label"]

groups = df["subject_id"]
# ==========================================
# CELL 11: Dataset Summary
# ==========================================

# Prepare features, labels, and groups
X = df.drop(
    ["label", "subject_id"],
    axis=1
)

y = df["label"].values

groups = df["subject_id"].values

print("========== DATASET SUMMARY ==========")

print("Feature Matrix Shape :", X.shape)
print("Labels Shape         :", y.shape)
print("Groups Shape         :", groups.shape)

print("\nSubjects:")
print(np.unique(groups))

print("\nNumber of Subjects:",
      len(np.unique(groups)))

print("\nClass Distribution:")

class_counts = pd.Series(y).value_counts().sort_index()

class_names = {
    0: "Non-Stress",
    1: "Stress"
}

for cls, count in class_counts.items():

    print(
        f"{class_names[cls]} ({cls}): "
        f"{count} windows "
        f"({count/len(y)*100:.2f}%)"
    )


print("\nFeature Summary:")
display(
    X.describe().T[
        ["mean", "std", "min", "max"]
    ]
)

print("\nTotal Features:",
      X.shape[1])


# LOSO Setup
logo = LeaveOneGroupOut()

print("\nLOSO Folds:",
      logo.get_n_splits(
          X,
          y,
          groups
      ))
# ==========================================
# Feature Groups for Ablation
# ==========================================

eda_features = [
    "eda_mean",
    "eda_std",
    "eda_slope",
    "scl_mean",
    "phasic_mean",
    "scr_count",
    "scr_amp_mean",
    "scr_rise_mean",
    "scr_recovery_mean"
]

hrv_features = [
    "hr",
    "rmssd",
    "sdnn",
    "pnn50",
    "ibi_mean",
    "ibi_std"
]

acc_features = [
    "imu_mag_mean",
    "imu_mag_std",
    "imu_energy",
    "imu_jerk_mean",
    "imu_jerk_std",
    "imu_var_x",
    "imu_var_y",
    "imu_var_z"
]

feature_sets = {
    "EDA": eda_features,

    "HRV": hrv_features,

    "EDA + HRV":
        eda_features + hrv_features,

    "EDA + HRV + ACC":
        eda_features + hrv_features + acc_features
}

print("Feature groups created.")
# ==========================================
# CELL 12: Random Forest LOSO Baseline
# ==========================================

rf_results = []

rf_true = []
rf_pred = []
rf_prob = []
rf_subjects = []

fold = 1

for train_idx, test_idx in logo.split(X, y, groups):

    test_subject = np.unique(groups[test_idx])[0]

    print(f"\nFold {fold} | Test Subject: {test_subject}")

    # Split data
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    # Standardization (fit only on training set)
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)

    X_test_scaled = scaler.transform(X_test)

    # Random Forest
    rf = RandomForestClassifier(
        n_estimators=300,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )

    rf.fit(
        X_train_scaled,
        y_train
    )

    # Predictions
    y_pred = rf.predict(X_test_scaled)

    y_proba = rf.predict_proba(X_test_scaled)[:, 1]

    # Metrics
    acc = accuracy_score(
        y_test,
        y_pred
    )

    prec = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    rec = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    auc = roc_auc_score(
        y_test,
        y_proba
    )

    print(
        f"Acc={acc:.3f} | "
        f"Prec={prec:.3f} | "
        f"Rec={rec:.3f} | "
        f"F1={f1:.3f} | "
        f"AUC={auc:.3f}"
    )

    rf_results.append({
        "Subject": test_subject,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "ROC_AUC": auc
    })

    rf_true.extend(y_test)

    rf_pred.extend(y_pred)

    rf_prob.extend(y_proba)

    rf_subjects.extend(
        [test_subject] * len(y_test)
    )

    fold += 1


# Results Table
rf_results_df = pd.DataFrame(rf_results)

print("\n===================================")
print("RANDOM FOREST LOSO SUMMARY")
print("===================================")

print(
    f"Accuracy : {rf_results_df['Accuracy'].mean()*100:.2f} ± "
    f"{rf_results_df['Accuracy'].std()*100:.2f}"
)

print(
    f"Precision: {rf_results_df['Precision'].mean()*100:.2f}"
)

print(
    f"Recall   : {rf_results_df['Recall'].mean()*100:.2f}"
)

print(
    f"F1 Score : {rf_results_df['F1'].mean()*100:.2f}"
)

print(
    f"ROC-AUC  : {rf_results_df['ROC_AUC'].mean()*100:.2f}"
)

display(rf_results_df)
XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=...
)
# ==========================================
# CELL 13: XGBoost LOSO Final Model
# ==========================================

xgb_results = []

xgb_true = []
xgb_pred = []
xgb_prob = []
xgb_subjects = []

fold = 1

for train_idx, test_idx in logo.split(X, y, groups):

    test_subject = np.unique(groups[test_idx])[0]

    print(f"\nFold {fold} | Test Subject: {test_subject}")

    # Split
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    # Standardize
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Handle imbalance
    n_nonstress = np.sum(y_train == 0)
    n_stress = np.sum(y_train == 1)

    scale_pos_weight = n_nonstress / n_stress

    # XGBoost
    xgb = XGBClassifier(
        objective='binary:logistic',

        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,

        subsample=0.8,
        colsample_bytree=0.8,

        scale_pos_weight=scale_pos_weight,

        random_state=42,

        eval_metric='logloss',

        n_jobs=-1
    )

    xgb.fit(
        X_train_scaled,
        y_train
    )

    # Predictions
    y_proba = xgb.predict_proba(
        X_test_scaled
    )[:, 1]

    y_pred = (y_proba >= 0.5).astype(int)

    # Metrics
    acc = accuracy_score(y_test, y_pred)

    prec = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    rec = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    auc = roc_auc_score(
        y_test,
        y_proba
    )

    print(
        f"Acc={acc:.3f} | "
        f"Prec={prec:.3f} | "
        f"Rec={rec:.3f} | "
        f"F1={f1:.3f} | "
        f"AUC={auc:.3f}"
    )

    # Store
    xgb_results.append({

        "Subject": test_subject,

        "Accuracy": acc,

        "Precision": prec,

        "Recall": rec,

        "F1": f1,

        "ROC_AUC": auc

    })

    xgb_true.extend(y_test)

    xgb_pred.extend(y_pred)

    xgb_prob.extend(y_proba)

    xgb_subjects.extend(
        [test_subject] * len(y_test)
    )

    fold += 1


# Convert to dataframe
xgb_results_df = pd.DataFrame(
    xgb_results
)

print("\n===================================")
print("XGBOOST LOSO SUMMARY")
print("===================================")

print(
    f"Accuracy : "
    f"{xgb_results_df['Accuracy'].mean()*100:.2f} ± "
    f"{xgb_results_df['Accuracy'].std()*100:.2f}"
)

print(
    f"Precision: "
    f"{xgb_results_df['Precision'].mean()*100:.2f}"
)

print(
    f"Recall   : "
    f"{xgb_results_df['Recall'].mean()*100:.2f}"
)

print(
    f"F1 Score : "
    f"{xgb_results_df['F1'].mean()*100:.2f}"
)

print(
    f"ROC-AUC  : "
    f"{xgb_results_df['ROC_AUC'].mean()*100:.2f}"
)

display(xgb_results_df)
# ==========================================
# LOSO Ablation Study
# ==========================================

ablation_results = []

for feature_name, selected_features in feature_sets.items():

    print("\n===================================")
    print("Testing:", feature_name)
    print("===================================")

    X_subset = df[selected_features]

    fold_metrics = []

    for train_idx, test_idx in logo.split(
        X_subset,
        y,
        groups
    ):

        X_train = X_subset.iloc[train_idx]
        X_test = X_subset.iloc[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        n_nonstress = np.sum(y_train == 0)
        n_stress = np.sum(y_train == 1)

        scale_pos_weight = (
            n_nonstress / n_stress
        )

        model = XGBClassifier(
            objective='binary:logistic',
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            eval_metric='logloss',
            n_jobs=-1
        )

        model.fit(
            X_train_scaled,
            y_train
        )

        y_prob = model.predict_proba(
            X_test_scaled
        )[:, 1]

        y_pred = (
            y_prob >= 0.5
        ).astype(int)

        fold_metrics.append({

            "Accuracy":
                accuracy_score(
                    y_test,
                    y_pred
                ),

            "Recall":
                recall_score(
                    y_test,
                    y_pred,
                    zero_division=0
                ),

            "F1":
                f1_score(
                    y_test,
                    y_pred,
                    zero_division=0
                ),

            "ROC_AUC":
                roc_auc_score(
                    y_test,
                    y_prob
                )
        })

    fold_df = pd.DataFrame(
        fold_metrics
    )

    ablation_results.append({

        "Features":
            feature_name,

        "Accuracy":
            fold_df["Accuracy"].mean() * 100,

        "Recall":
            fold_df["Recall"].mean() * 100,

        "F1":
            fold_df["F1"].mean() * 100,

        "ROC_AUC":
            fold_df["ROC_AUC"].mean() * 100
    })

ablation_df = pd.DataFrame(
    ablation_results
)

display(ablation_df)
print("xgb_true:", len(xgb_true))
print("xgb_prob:", len(xgb_prob))
print("xgb_subjects:", len(xgb_subjects))
# ==========================================
# MHSI Validation DataFrame
# ==========================================

mhsi_df = pd.DataFrame({

    "Subject": xgb_subjects,

    "True_Label": xgb_true,

    "Stress_Probability": xgb_prob

})

mhsi_df["MHSI"] = (
    mhsi_df["Stress_Probability"] * 100
)

mhsi_df["Class"] = mhsi_df[
    "True_Label"
].map({

    0: "Non-Stress",

    1: "Stress"

})

print(mhsi_df.head())

print("\nShape:", mhsi_df.shape)
# ==========================================
# MHSI Summary Statistics
# ==========================================

summary = mhsi_df.groupby(
    "Class"
)["MHSI"].describe()

display(summary)
# ==========================================
# MHSI Boxplot
# ==========================================

plt.figure(figsize=(7,5))

sns.boxplot(
    data=mhsi_df,
    x="Class",
    y="MHSI"
)

plt.title(
    "Distribution of MHSI Across Stress States"
)

plt.ylabel("MHSI")

plt.xlabel("")

plt.show()
# ==========================================
# Statistical Validation of MHSI
# ==========================================

from scipy.stats import mannwhitneyu

nonstress = mhsi_df[
    mhsi_df["True_Label"] == 0
]["MHSI"]

stress = mhsi_df[
    mhsi_df["True_Label"] == 1
]["MHSI"]

stat, p = mannwhitneyu(
    nonstress,
    stress,
    alternative="two-sided"
)

print("Mann-Whitney U Statistic:", stat)

print("p-value:", p)
# ==========================================
# Correct Effect Size (Rank-Biserial Correlation)
# ==========================================

n1 = len(nonstress)
n2 = len(stress)

rank_biserial = (
    (2 * stat) / (n1 * n2)
) - 1

print(
    "Rank-Biserial Correlation:",
    round(rank_biserial, 4)
)

# Interpretation

if abs(rank_biserial) < 0.1:
    interpretation = "Negligible"

elif abs(rank_biserial) < 0.3:
    interpretation = "Small"

elif abs(rank_biserial) < 0.5:
    interpretation = "Moderate"

else:
    interpretation = "Large"

print("Effect Size Interpretation:", interpretation)
!pip install -q catboost
# ==========================================
# CatBoost LOSO Evaluation
# ==========================================

from catboost import CatBoostClassifier

cat_results = []

cat_true = []
cat_pred = []
cat_prob = []
cat_subjects = []

fold = 1

for train_idx, test_idx in logo.split(X, y, groups):

    test_subject = np.unique(groups[test_idx])[0]

    print(f"\nFold {fold} | Test Subject: {test_subject}")

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    n_nonstress = np.sum(y_train == 0)
    n_stress = np.sum(y_train == 1)

    class_weights = [
        1,
        n_nonstress / n_stress
    ]

    cat = CatBoostClassifier(

        iterations=300,

        depth=6,

        learning_rate=0.05,

        loss_function="Logloss",

        eval_metric="AUC",

        class_weights=class_weights,

        random_seed=42,

        verbose=False

    )

    cat.fit(
        X_train_scaled,
        y_train
    )

    y_proba = cat.predict_proba(
        X_test_scaled
    )[:,1]

    y_pred = (
        y_proba >= 0.5
    ).astype(int)

    acc = accuracy_score(
        y_test,
        y_pred
    )

    prec = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    rec = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    auc = roc_auc_score(
        y_test,
        y_proba
    )

    print(
        f"Acc={acc:.3f} | "
        f"Prec={prec:.3f} | "
        f"Rec={rec:.3f} | "
        f"F1={f1:.3f} | "
        f"AUC={auc:.3f}"
    )

    cat_results.append({

        "Subject": test_subject,

        "Accuracy": acc,

        "Precision": prec,

        "Recall": rec,

        "F1": f1,

        "ROC_AUC": auc

    })

    cat_true.extend(y_test)

    cat_pred.extend(y_pred)

    cat_prob.extend(y_proba)

    cat_subjects.extend(
        [test_subject] * len(y_test)
    )

    fold += 1

cat_results_df = pd.DataFrame(cat_results)

print("\n===================================")
print("CATBOOST LOSO SUMMARY")
print("===================================")

print(
    f"Accuracy : {cat_results_df['Accuracy'].mean()*100:.2f} ± "
    f"{cat_results_df['Accuracy'].std()*100:.2f}"
)

print(
    f"Precision: {cat_results_df['Precision'].mean()*100:.2f}"
)

print(
    f"Recall   : {cat_results_df['Recall'].mean()*100:.2f}"
)

print(
    f"F1 Score : {cat_results_df['F1'].mean()*100:.2f}"
)

print(
    f"ROC-AUC  : {cat_results_df['ROC_AUC'].mean()*100:.2f}"
)

display(cat_results_df)

/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/wesad_readme.pdf
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14.pkl
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_quest.csv
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_respiban.txt
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_readme.txt
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_E4_Data/HR.csv
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_E4_Data/IBI.csv
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_E4_Data/BVP.csv
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14_E4_Data/EDA.csv
/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-da